# Global CLIP Evaluation on MVTec AD
Automated zero-shot evaluation across all 15 categories.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Google Colab.")


In [ ]:
!pip install "anomalib[vlm,clip]" pandas open_clip_torch

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/dvydinh/ralm_industrial_anomaly_detection.git"
REPO_DIR = "/content/ralm_industrial_anomaly_detection"

try:
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    sys.path.insert(0, REPO_DIR)
except Exception as e:
    print("Could not clone repo, assuming running locally.")
    sys.path.insert(0, "..")


In [ ]:
import pandas as pd
from anomalib.models import WinClip
from anomalib.engine import Engine
from anomalib.data import MVTecAD
import torch

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
CATEGORIES = [
    'bottle', 'cable', 'capsule', 'carpet', 'grid',
    'hazelnut', 'leather', 'metal_nut', 'pill', 'screw',
    'tile', 'toothbrush', 'transistor', 'wood', 'zipper'
]

DATA_ROOT = "/content/drive/MyDrive/ralm/data/mvtec_anomaly_detection"
EVAL_BATCH_SIZE = 128
NUM_WORKERS = 8

all_results = []
engine = Engine()

for category in CATEGORIES:
    print(f"\n{'='*50}")
    print(f"Evaluating category: {category.upper()}")
    print(f"{'='*50}")
    
    datamodule = MVTecAD(
        root=DATA_ROOT,
        category=category,
        eval_batch_size=EVAL_BATCH_SIZE,
        num_workers=NUM_WORKERS
    )
    
    # Filter Anomalib test set to match RAML exact subset (20% test split)
    from raml.data.mvtec_dataset import MVTecDataset
    raml_test_ds = MVTecDataset(
        data_dir=DATA_ROOT,
        categories=[category],
        split="test",
        use_test_anomalies=True,
        train_ratio=0.8,
        seed=42
    )
    import os
    allowed_paths = {os.path.abspath(item["path"]) for item in raml_test_ds.samples}
    
    # Monkey-patch setup to ensure the filter applies when Engine calls setup()
    original_setup = datamodule.setup
    def custom_setup(stage=None):
        original_setup(stage)
        if stage == "test":
            df = datamodule.test_data.samples
            df["abs_path"] = df["image_path"].apply(os.path.abspath)
            datamodule.test_data.samples = df[df["abs_path"].isin(allowed_paths)].copy().reset_index(drop=True)
            
    datamodule.setup = custom_setup
    
    model = WinClip(class_name=category)
    results = engine.test(model=model, datamodule=datamodule)
    
    if results:
        res = results[0] if isinstance(results, list) else results
        res["category"] = category
        all_results.append(res)

if all_results:
    df = pd.DataFrame(all_results)
    display(df)


In [ ]:
import os
import json

SAVE_DIR = "/content/drive/MyDrive/ralm" if os.path.exists("/content/drive/MyDrive/ralm") else "../results"
os.makedirs(os.path.join(SAVE_DIR, "metrics"), exist_ok=True)

if all_results:
    per_category = {}
    macro_metrics = {"image_AUROC": 0, "image_F1Score": 0, "pixel_AUROC": 0, "pixel_F1Score": 0}
    
    for res in all_results:
        cat = res["category"]
        cat_dict = {
            "image_AUROC": res.get("image_AUROC", 0) * 100,
            "image_F1Score": res.get("image_F1Score", 0) * 100,
            "pixel_AUROC": res.get("pixel_AUROC", 0) * 100,
            "pixel_F1Score": res.get("pixel_F1Score", 0) * 100
        }
        per_category[cat] = cat_dict
        for k in macro_metrics:
            macro_metrics[k] += cat_dict[k]
            
    for k in macro_metrics:
        macro_metrics[k] /= len(all_results)
        
    result_json = {
        "method": "Anomalib_WinCLIP",
        "macro_metrics": {k: round(v, 2) for k, v in macro_metrics.items()},
        "per_category": {k: {m_k: round(m_v, 2) for m_k, m_v in v.items()} for k, v in per_category.items()}
    }
    
    json_path = os.path.join(SAVE_DIR, "metrics", "result_winclip.json")
    with open(json_path, "w") as f:
        json.dump(result_json, f, indent=2)
        
    csv_path = os.path.join(SAVE_DIR, "metrics", "result_winclip.csv")
    rows = [{"Category": k, **v} for k, v in per_category.items()]
    df_save = pd.DataFrame(rows)
    df_save.loc[len(df_save)] = ["MEAN", *[macro_metrics[m] for m in df_save.columns if m != "Category"]]
    df_save.to_csv(csv_path, index=False)
    print(f"WinCLIP results strictly saved to {SAVE_DIR}/metrics/")
